# 03 - Famille d'attaques non vue (split de généralisation)

**Projet :** Détection de prompt injection et jailbreak — **Membre 1 (Données & Baseline)**

## Objectif
Le cahier des charges demande de *retirer une famille d'attaques de l'entraînement, puis de l'utiliser uniquement au test*.
Ce notebook construit les 4 fichiers finaux utilisés par toute l'équipe :

| Fichier | Contenu | Sert à |
|---|---|---|
| `train_seen.csv` | train **sans** la famille retirée | entraîner |
| `val_seen.csv` | validation **sans** la famille retirée | régler / choisir |
| `test_seen.csv` | test **sans** la famille retirée | mesurer sur des attaques connues |
| `test_unseen_family.csv` | **toute** la famille retirée + prompts légitimes du test | mesurer la généralisation |

## Choix de la famille retirée : `jailbreak`
Familles disponibles dans le dataset (colonne `label_category`) : `direct_injection`, `indirect_injection`, `jailbreak`.
On retire `jailbreak` car (1) c'est une famille citée dans le titre du projet, (2) elle est assez grande (~1000 exemples),
(3) elle est stylistiquement très différente des injections directes (textes longs, jeux de rôle).
Pour changer de famille, il suffit de modifier la variable `HELD_OUT_FAMILY` ci-dessous et de relancer tout le notebook.

In [ ]:
import os
import pandas as pd

# Se placer à la racine du dépôt (là où se trouve le dossier data/)
while not os.path.exists("data/processed") and os.getcwd() != "/":
    os.chdir("..")
print("Racine du dépôt :", os.getcwd())

## 1. Paramètres

In [ ]:
HELD_OUT_FAMILY = "jailbreak"   # famille retirée de l'entraînement
IN_DIR = "data/processed"
OUT_DIR = "data/processed"
os.makedirs("results", exist_ok=True)

## 2. Chargement des fichiers nettoyés (notebook 02)

In [ ]:
train = pd.read_csv(f"{IN_DIR}/train_clean.csv")
val   = pd.read_csv(f"{IN_DIR}/validation_clean.csv")
test  = pd.read_csv(f"{IN_DIR}/test_clean.csv")

all_df = pd.concat([train, val, test], ignore_index=True)
print("Tailles :", len(train), len(val), len(test), "->", len(all_df))
display(pd.crosstab(all_df["split"], all_df["label_category"], margins=True))

## 3. Construction des 4 fichiers

Règles :
1. Une ligne appartient à la famille retirée si `label_category == HELD_OUT_FAMILY`.
2. `train_seen`, `val_seen`, `test_seen` = leurs ensembles d'origine **moins** la famille retirée.
3. `test_unseen_family` = **toutes** les lignes de la famille retirée (venant de train, validation et test : aucune ne sert à l'entraînement)
   **+** les prompts légitimes du test d'origine. Les légitimes sont indispensables : sans eux on ne pourrait pas mesurer le taux de faux positifs.

In [ ]:
is_held = all_df["label_category"] == HELD_OUT_FAMILY

train_seen = all_df[(all_df["split"] == "train") & ~is_held].copy()
val_seen   = all_df[(all_df["split"] == "validation") & ~is_held].copy()
test_seen  = all_df[(all_df["split"] == "test") & ~is_held].copy()

held_rows   = all_df[is_held].copy()
benign_test = all_df[(all_df["split"] == "test") & (all_df["label_binary"] == 0)].copy()
test_unseen = pd.concat([held_rows, benign_test], ignore_index=True)

# Colonne pratique : la "famille" (benign pour les prompts légitimes)
for d in (train_seen, val_seen, test_seen, test_unseen):
    d["family"] = d["label_category"]

print("train_seen :", len(train_seen))
print("val_seen   :", len(val_seen))
print("test_seen  :", len(test_seen))
print("test_unseen:", len(test_unseen))

## 4. Vérifications automatiques (`assert`)

Un `assert` arrête le notebook avec une erreur si la condition est fausse : c'est notre filet de sécurité contre les fuites de données.

In [ ]:
norm = lambda s: s.astype(str).str.strip().str.lower()

# 1) La famille retirée est ABSENTE de train/val/test_seen
for name, d in [("train_seen", train_seen), ("val_seen", val_seen), ("test_seen", test_seen)]:
    assert (d["label_category"] == HELD_OUT_FAMILY).sum() == 0, f"{name} contient la famille retirée !"

# 2) Aucun texte commun entre train_seen, val_seen et test_seen
t, v, s = set(norm(train_seen.text)), set(norm(val_seen.text)), set(norm(test_seen.text))
assert len(t & v) == 0 and len(t & s) == 0 and len(v & s) == 0, "Fuite entre train/val/test !"

# 3) Aucun texte malveillant du test_unseen n'est dans le train ou la validation
u_mal = set(norm(test_unseen[test_unseen.label_binary == 1].text))
assert len(u_mal & t) == 0 and len(u_mal & v) == 0, "Des attaques du test_unseen sont dans le train/val !"

# 4) Aucun texte légitime du test_unseen n'est dans le train
u_ben = set(norm(test_unseen[test_unseen.label_binary == 0].text))
assert len(u_ben & t) == 0, "Des textes légitimes du test_unseen sont dans le train !"

# 5) test_unseen contient les deux classes
assert set(test_unseen.label_binary.unique()) == {0, 1}

# 6) Pas de texte vide ni de doublon exact à l'intérieur de chaque fichier
for name, d in [("train_seen", train_seen), ("val_seen", val_seen), ("test_seen", test_seen), ("test_unseen", test_unseen)]:
    assert d["text"].astype(str).str.strip().ne("").all(), f"texte vide dans {name}"
    assert norm(d.text).duplicated().sum() == 0, f"doublon dans {name}"

print("Toutes les vérifications sont passées ✓")

## 5. Résumé des fichiers

In [ ]:
files = {"train_seen": train_seen, "val_seen": val_seen, "test_seen": test_seen, "test_unseen_family": test_unseen}
rows = []
for name, d in files.items():
    rows.append({
        "fichier": name + ".csv",
        "lignes": len(d),
        "legitimes (0)": int((d.label_binary == 0).sum()),
        "malveillants (1)": int((d.label_binary == 1).sum()),
        "% malveillants": round(100 * d.label_binary.mean(), 1),
        "familles": ", ".join(sorted(d.family.unique())),
    })
summary = pd.DataFrame(rows)
display(summary)
summary.to_csv("results/split_summary.csv", index=False)

## 6. Sauvegarde

In [ ]:
cols = ["id", "text", "label_binary", "family", "source", "language", "split"]
for name, d in files.items():
    d[cols].to_csv(f"{OUT_DIR}/{name}.csv", index=False, encoding="utf-8")
    print("✓", f"{OUT_DIR}/{name}.csv", len(d), "lignes")

## Conclusion
- La famille **`jailbreak`** est complètement absente de `train_seen`, `val_seen` et `test_seen`.
- `test_unseen_family.csv` permet de mesurer le **taux de détection** sur cette famille jamais vue **et** le **taux de faux positifs** sur des prompts légitimes.
- **Limite à signaler dans le rapport :** presque tous les jailbreaks viennent d'une même source (`trustailab/in-the-wild-jailbreak-prompts`). Retirer la famille revient donc aussi à retirer un *style* de rédaction.
- Les fichiers `train_seen / val_seen / test_seen / test_unseen_family` sont à utiliser par les Membres 2 et 3.